# 05 — Convolutional Neural Networks (CNN)

This notebook trains convolutional networks **from scratch** on the H&M images. Convolutions share weights across positions and preserve the spatial structure that the Dense model discards, so a CNN can be seen as a learned **feature extractor followed by a classifier**.

Starting from a small Conv–ReLU–Pool stack, the model is progressively extended with batch normalisation, dropout, global average pooling and **residual (skip) connections** in the spirit of ResNet. Feature maps and filters are visualised to understand what each layer learns about garments (edges, textures, silhouettes).

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Vía A · CNN desde cero y posible atención

CNN es una arquitectura, mientras que scratch describe cómo inicializamos sus pesos. Aquí se estudia la convolución espacial en imágenes; el notebook 07 reutilizará pesos preentrenados como otra vía. Que la CNN pequeña gane o pierda frente a Dense importa menos que explicar qué aporta su estructura local y el peso compartido.

La pista «CNN + attention layer» puede interpretarse como una extensión: añadir atención sobre canales o posiciones, o transformar posiciones de un mapa de características en tokens. No es un requisito para que una red sea CNN. Una CNN1D sobre word embeddings pertenece a la rama NLP opcional de 11 y no es este experimento visual. Una híbrida CNN + LSTM/GRU también es posible, pero no convierte las unidades recurrentes en convoluciones. Si se añade atención, comparar con la misma CNN sin ella mediante una ablación, sin prometer mejoras. [Fundamento del mecanismo de atención](https://arxiv.org/abs/1706.03762).


## Protocolo revisado antes del entrenamiento · 20/11/2026

La referencia común es `config/experimental_protocol_v1.json`. Es un diseño fijado antes de entrenar las redes, no una lista de resultados obtenidos. La versión equilibrada de 18.000 fotos se conserva como experimento exploratorio. Sus métricas describen las doce clases seleccionadas con ese muestreo; no estiman automáticamente el comportamiento sobre el catálogo original.

**Por defecto:** entropía cruzada sin pesos, sin aumentos y con normalización de train a la resolución de entrada. Pesos de clase y aumentos se activarán como ablaciones explícitas. Los pesos guardados en 02 son una opción calculada, no una instrucción de aplicarlos siempre. Cada experimento registrará resolución, preprocesamiento, IDs de productos, configuración, semilla y presupuesto.

**Métricas fijadas:** macro-F1 primaria; accuracy secundaria; precisión, recall, F1 y soporte por clase; matriz de confusión en recuentos y normalizada por filas. Comparar parámetros entrenables, tiempo de entrenamiento, latencia por imagen, memoria pico y hardware. Selección por macro-F1 de validation; early stopping por pérdida de validation; tres semillas (42, 123 y 2026). El test permanece cerrado hasta congelar decisiones; intervalos finales mediante remuestreo de grupos de producto, no imágenes independientes. Los baselines deterministas no necesitan simular tres entrenamientos diferentes.

### Hipótesis que justifican esta CNN

H1 compara con Dense a RGB64 y parámetros aproximadamente iguales; el resultado se expresará como diferencia de macro-F1 por semilla y por clase, sin prometer una mejora mínima inventada. H2 utiliza una CNN fija: normalización sí/no y aumentos sí/no, cambiando un factor cada vez. H4 contrasta entropía cruzada sin pesos con los pesos de train próximos a uno. La configuración inicial no combina automáticamente normalización, aumentos, BatchNorm y pesos para luego atribuirles conjuntamente una mejora.

La referencia aplicada a 128×128 se informará aparte de la comparación controlada a 64×64. Las clases que supuestamente se benefician más de la estructura espacial se identificarán con diferencias por clase en validation, no a partir de intuiciones ni del test.


## Objectives

- Build a baseline CNN (Conv → ReLU → Pool blocks + Dense softmax head) with the shared data pipeline.
- Compare it with the Dense model in accuracy, macro F1, parameters and training time.
- Add batch normalisation, dropout and global average pooling and observe their effect on training stability.
- Implement a small residual block and test whether skip connections help deeper networks train.
- Visualise learned filters and feature maps for example garments of several classes.
- Select the best from-scratch CNN for the final comparison.

## Preview of the work

1. **Setup.** loaders at a CNN resolution (e.g. 128×128 RGB) and training utilities from `src/training/`.
2. **Baseline CNN.** architecture, receptive field, number of parameters.
3. **Training and learning curves.** comparison with the Dense baseline.
4. **Architectural improvements.** BatchNorm, dropout, global average pooling, depth.
5. **Residual blocks.** a mini-ResNet and the link between skip connections and gradient flow.
6. **Interpretation.** filters, feature maps per layer, misclassified examples.
7. **Validation results and conclusions.** best from-scratch CNN.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Loaders and configuration from notebook 02; Dense results from notebook 04 |
| **Outputs** | Metrics, histories, checkpoints and feature-map figures in `results/` |
| **Course material** | `PRDL_MLLB_CNN.pptx`, `PRDL_MLLB_RestNets_to_NeuralODE.pptx`, `MLLB_CNN_MNIST_in_Keras.ipynb`, `Gemini_MNIST_Pytorch_featuremaps.ipynb` |